# plate-redactor — Phase 4: TFLite export & quantisation

Exports the Phase-2 checkpoint (`models/best.pt`) to a quantised `.tflite`,
verifies its I/O signature against the README §2 contract, benchmarks CPU
latency, and re-runs the Phase-3 recall gate against the exported model.

**Quantisation: fp16 first** (halves size, no calibration, negligible accuracy
loss). Fall back to int8 only if fp16 exceeds 8 MB or recall drops > 2 pp.

Runs on Kaggle or Colab (CPU is fine). Needs the trained `models/best.pt` from
Phase 2 (release asset / Kaggle output / Drive — **never committed**).

## 1. Install dependencies

The `[export]` extra pulls `ultralytics` (export) + `tensorflow` (TFLite runtime
for the raw-interpreter verify/benchmark). Ultralytics auto-installs the export
toolchain (`onnx`, `onnx2tf`, …) on first use.

In [ ]:
%pip install -q ultralytics tensorflow

import ultralytics
ultralytics.checks()

## 2. Get the code + detect platform

In [ ]:
import os, sys, subprocess
from pathlib import Path

ON_KAGGLE = Path('/kaggle').exists()
ON_COLAB = (not ON_KAGGLE) and ('google.colab' in sys.modules or Path('/content').exists())
print('Kaggle:', ON_KAGGLE, '| Colab:', ON_COLAB)

REPO_URL = 'https://github.com/Andre-Ehret/plate-redactor.git'
BRANCH = 'main'
WORK = Path('/kaggle/working') if ON_KAGGLE else Path('/content')
REPO = WORK / 'plate-redactor'

if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, str(REPO)], check=True)
else:
    # Force to latest main so code fixes land on a plain re-run.
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', f'origin/{BRANCH}'], check=True)

%pip install -q -e {str(REPO)}
os.chdir(REPO)
head = subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip()
print('cwd:', os.getcwd(), '| HEAD:', head)

## 3. Get the checkpoint (`models/best.pt`)

Point `MODEL` at the Phase-2 checkpoint. Pick the option that matches where you
stored it (Kaggle Dataset / GitHub release asset / Google Drive). It is copied to
`models/best.pt` (the script default).

In [ ]:
import shutil
MODEL = REPO / 'models' / 'best.pt'
MODEL.parent.mkdir(parents=True, exist_ok=True)

# --- Option A: attached Kaggle Dataset (edit the path) -------------------------
# src = Path('/kaggle/input/plate-detector-best/best.pt')
# shutil.copy2(src, MODEL)

# --- Option B: download a GitHub release asset (edit the URL) -------------------
# ASSET_URL = 'https://github.com/Andre-Ehret/plate-redactor/releases/download/v0.1/best.pt'
# subprocess.run(['curl', '-L', '-o', str(MODEL), ASSET_URL], check=True)

# --- Option C: Google Drive (Colab) --------------------------------------------
# from google.colab import drive; drive.mount('/content/drive')
# shutil.copy2('/content/drive/MyDrive/plate-redactor/best.pt', MODEL)

assert MODEL.exists(), f'{MODEL} missing — fill in one of the options above.'
print('checkpoint:', MODEL, f'({MODEL.stat().st_size/1e6:.1f} MB)')

## 4. Export to TFLite (fp16)

fp16, `imgsz=320` (locked in Phase 2), NMS baked into the graph by default.
Writes `models/plate-detector.tflite`, the versioned release artefact, and
`models/export_meta.json`. If the size exceeds 8 MB, re-run with
`--quantisation int8 --data data/synthetic/data.yaml`.

In [ ]:
subprocess.run([
    sys.executable, 'src/export/export.py',
    '--model', str(MODEL), '--imgsz', '320', '--quantisation', 'fp16',
], check=True)

## 5. Verify the I/O signature + NMS location

Loads the `.tflite` with the raw runtime (what the app uses), prints input/output
tensors, states whether NMS is baked in or needs post-processing, and drafts
`models/export_report.md`.

In [ ]:
subprocess.run([sys.executable, 'src/export/verify_output.py'], check=True)

## 6. Build the test set + benchmark CPU latency

The benchmark samples val images and the recall check needs the hard-case test
set (seed 999). Generate a small synthetic set for the benchmark, then the Phase-3
test set. Add `--backgrounds /kaggle/input/<bg-dataset>` to match the checkpoint's
training backgrounds for a fair recall test.

In [ ]:
# Small val set for the latency benchmark (skip if data/synthetic already exists).
if not (REPO / 'data' / 'synthetic' / 'images' / 'val').exists():
    subprocess.run([
        sys.executable, '-m', 'plate_redactor.generator.generate',
        '--n', '200', '--seed', '42', '--out', 'data/synthetic',
    ], check=True)

subprocess.run([sys.executable, 'src/export/benchmark.py', '--n', '20'], check=True)

## 7. Recall regression (hard gate ≥ 0.90)

Re-runs the Phase-3 evaluation against the **exported `.tflite`** (Ultralytics
loads it through its TFLite runtime — same pre/post-processing as the fp32 run).
Recall must stay ≥ 0.90; if it drops > 2 pp vs. Phase 3, switch to int8.

In [ ]:
subprocess.run([
    sys.executable, 'src/eval/generate_test_set.py',
    '--out', 'data/test_hard', '--seed', '999',
    '--per-subset', '100', '--per-orientation', '50',
], check=True)

subprocess.run([
    sys.executable, 'src/eval/evaluate.py',
    '--model', 'models/plate-detector.tflite',
    '--data', 'data/test_hard', '--conf', '0.25', '--imgsz', '320',
], check=True)

## 8. Re-draft the report with all numbers + inspect

Re-run `verify_output.py` now that benchmark + eval JSON exist so the report folds
in latency and recall. Then review `models/export_report.md` — confirm the NMS
note and paste the Phase-3 fp32-baseline recall + delta by hand before committing.

In [ ]:
import json
from IPython.display import Markdown, display

subprocess.run([sys.executable, 'src/export/verify_output.py'], check=True)

meta = json.loads((REPO / 'models' / 'export_meta.json').read_text())
bench = json.loads((REPO / 'models' / 'benchmark_results.json').read_text())
ev = json.loads((REPO / 'models' / 'eval_results.json').read_text())
print('size MB:', meta['exported_size_mb'], '(≤ 8:', meta['size_ok'], ')')
print('p95 ms :', bench['p95_ms'], '(≤ 500:', bench['p95_ok'], ')')
print('recall :', ev['overall']['recall'], '(gate passed:', ev['gates']['passed'], ')')

display(Markdown((REPO / 'models' / 'export_report.md').read_text()))

## 9. Save the release artefact

`plate-detector-v0.1.0.tflite` is the versioned model — download it and upload as
a **GitHub Release asset** (it is gitignored, never committed). On Kaggle it is
already under `models/` in the working dir; on Colab copy it to Drive.

In [ ]:
artefact = next((REPO / 'models').glob('plate-detector-v*.tflite'))
print('release artefact:', artefact, f'({artefact.stat().st_size/1e6:.2f} MB)')

# Colab: persist to Drive (uncomment after mounting in cell 3, Option C)
# shutil.copy2(artefact, '/content/drive/MyDrive/plate-redactor/' + artefact.name)
# Kaggle: it is already in /kaggle/working/plate-redactor/models/ — download from the UI.